# 67 · PT pathways: T_spatial normalized by specimen shape variation

**The question.** Notebook 66 found that limma's cameraPR and gseapy's GSEA on `T_spatial` are not
specific: under the balanced relabelings they call 29–146 pathways, and 12–21% of their relabeled
p-values are ≤ 0.05. It suggested two reasons:
- `T_spatial` grows with counts;
- it treats structures as independent, so shape differences between individual specimens count as
  group differences.

The user asked whether `T_spatial` should be normalized. This notebook divides each gene's
`T_spatial` by that gene's own specimen-to-specimen shape variation and asks whether the packaged set
tests become specific.

**The frozen protocol.** Everything below follows `docs/results/pt-gene-model-nb.md`, section
"Notebook 67 protocol: T_spatial normalized by specimen shape variation". It was committed in
13f0148, before any fit.
- **Numerator.** `T_spatial`, unchanged: the NB likelihood ratio (LR) of M_full against M_full
  without group × basis, df₁ = 6 (`pathway_pipelines.standard_gene_model`).
- **Denominator.** LR_spec = D(M_full) − D(M_spec). M_spec adds specimen-within-group × basis to
  M_full, so every specimen gets its own smooth shape. df₂ = 12 for the condition split and 6 for a
  relabeling.
- **Moderation.** s² = LR_spec / df₂ is moderated by `limma::squeezeVar`, with log mean count per
  structure as covariate (primary). Sensitivities: no covariate, and `robust = TRUE`.
- **Statistic.** F = (`T_spatial` / 6) / s²_post with df (6, df₂ + d₀), converted to
  z = Φ⁻¹(1 − p).
- **Pathway tests on z** (gene sets of 10–300 genes, as in notebook 66):
  - `limma::cameraPR` (`inter.gene.cor` 0.01, `use.ranks` FALSE; a call is Up with FDR ≤ 0.05):
    **the primary**;
  - gseapy prerank on z;
  - ORA with `gseapy.enrich` on the genes with BH ≤ 0.05, against all tested genes;
  - sensitivities: cameraPR on rank-normal scores of the F; cameraPR without the genes that have zero
    counts in any specimen.
- **Pass rule** (notebook 66's, unchanged): NCDR < 0.25 and each relabeling below 5% of tested
  pathways; pooled share of relabeled one-sided p ≤ 0.05 at most 0.075.
- **Decision.** If the primary passes on both datasets, it becomes the user's candidate replacement
  for the joint test as primary; the user decides. Otherwise the result is reported as is.

**Two datasets**, built exactly as in notebook 66:

| Dataset | Structures | Genes | Pathways | Reference | Case | Coordinate |
|---|---|---|---|---|---|---|
| Human vs mouse PT | 12,272 | 11,071 | 1,513 | 2 control mice | 2 cortex sections of 1 human donor | SCF13 (notebook 13's scFates) |
| AKI vs control mouse PT | 16,991 | 9,107 | 1,429 | 2 control mice | 2 AKI mice | mouse-only PT DPT (mouse_only_v5) |

Each dataset is analysed three times: the **condition split** and the two **balanced relabelings**,
which pair one case specimen with one reference specimen. A relabeling has no condition difference
left to find, so its calls measure specimen-to-specimen noise.

**Sections:**
1. Setup, inputs and the redundancy clusters.
2. The numerator and the specimen-shape denominator.
3. The moderated F.
4. Pathway tests on z.
5. The method tables.
6. Gene-level information checks.
7. Overlap with notebook 66's methods.
8. What the normalization changed at gene level, with worked examples.
9. The pathways the primary calls.
10. Readings of the protocol.
11. Summary.
12. Run record.

**Assumptions.** Each coordinate is treated as exact. Specimens, not structures, are the replicates.
The two human "replicates" are two sections of one donor. With two specimens per group every result
is descriptive.

## 1 · Setup

In [ ]:
import argparse
import json
import os
import sys
import time
import warnings
from pathlib import Path

NOTEBOOK_START = time.perf_counter()
start = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
project = next(p for p in (start, *start.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
coordinate_file = results_root / 'pt_reconstruction_v2' / 'primary_pt_coordinate.csv'
upstream13 = results_root / 'minimal_pt_scfates'
mouse_v5 = results_root / 'mouse_only_v5'
universe_file = results_root / 'pt_pathway_remodeling_scfates' / 'gene_statistics.csv'      # notebook 45's universe
nb45_cache = results_root / 'pt_revision_method' / 'stage_cache'                            # read only, guards
nb65_genes = results_root / 'pt_gene_model_comparison' / 'tables' / 'gene_statistics_gaussian_and_nb.csv'  # read only, guard
nb66 = results_root / 'pt_pathway_pipelines_compared'     # read only: comparison rows, call sets, redundancy clusters, guards
output = results_root / 'pt_normalized_t_spatial'
tables, figures = output / 'tables', output / 'figures'
for folder in (tables, figures):
    folder.mkdir(parents=True, exist_ok=True)
DATASETS = ['human_vs_mouse', 'aki_vs_control']
LIBRARIES = ('Reactome_2022', 'MSigDB_Hallmark_2020', 'KEGG_2019_Mouse')
REQUIRED = [coordinate_file, upstream13 / 'cross_species_pt_scfates.h5ad', upstream13 / 'ortholog_map_used.csv',
            universe_file, mouse_v5 / 'all_mouse_tubules_scanpy_dpt.h5ad', nb66 / 'tables' / 'method_table.csv',
            *[nb66 / 'tables' / f'{kind}_{ds}.csv' for kind in ('redundancy_clusters', 'pathway_detail') for ds in DATASETS],
            *[data_root / 'mouse_vs_human' / 'pathway_gene_sets' / f'{lib}.json' for lib in LIBRARIES]]
for path in REQUIRED:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path} (run notebooks 13, 36 and 66 and the mouse_only_v5 workflow first)')

NOTEBOOK_LOGIC_VERSION = '67.normalized.1'   # Bump when a cached calculation changes.
ALPHA, NCDR_MAX, NULL_RATE = .05, .25, .05      # discovery level and notebook 31's relabeling rule
CALIBRATION_MAX = 1.5 * ALPHA                    # notebook 61's criterion 2: share of relabeled p <= 0.05
CAMERA_COR = .01                                 # limma's default inter.gene.cor (protocol)
GSEA_PERMUTATIONS, GSEA_SEED = 1000, 12          # as notebook 66
BASIS_DF, GRID_POINTS = 6, 101                   # the shared gene model
DF1 = 6                                          # numerator: group x basis
DF2 = {'condition': 12, 'relabeling': 6}         # denominator: specimen-within-group x basis (asserted from the designs)
GENE_FDR = .05                                   # ORA list: genes with BH <= 0.05 on the moderated F
ABSENT = .01                                     # nearly absent: detected in < 1% of one group's structures
SET_SIZES = (10, 300)
TOP_SHARE = .10                                  # section 8: the top 10% of genes by each statistic
N_JOBS = int(os.environ.get('PSEUDOSPACE_N_JOBS', '36'))   # parallel workers; never part of a cache key

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from importlib.metadata import version
from IPython.display import display

MM = 1 / 25.4
INK, MUTED, LIGHT = '#0b0b0b', '#52514e', '#d9d8d4'
A_COLOR, B_COLOR, REF_COLOR = '#2a78d6', '#eb6834', '#8a8984'   # notebook 66: method A, method B, the reference test
N_COLOR = '#1b8a5a'                                              # this notebook: the normalized statistic
CASE_COLOR, REFERENCE_COLOR = '#e34948', '#2a78d6'               # specimen curves (notebook 66's diverging ends)
plt.rcParams.update({
    'font.family': ['Liberation Sans', 'DejaVu Sans'], 'font.size': 6.5, 'axes.titlesize': 6.5,
    'axes.titleweight': 'bold', 'axes.labelsize': 6.5, 'xtick.labelsize': 6, 'ytick.labelsize': 6,
    'legend.fontsize': 6, 'legend.frameon': False, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.linewidth': .6, 'xtick.major.width': .6, 'ytick.major.width': .6, 'xtick.major.size': 2.5,
    'ytick.major.size': 2.5, 'lines.linewidth': 1.0, 'pdf.fonttype': 42, 'svg.fonttype': 'none', 'savefig.dpi': 300})
pd.set_option('display.width', 240, 'display.max_columns', 40, 'display.max_colwidth', 70, 'display.max_rows', 80)


def save(fig, stem):
    for suffix in ('png', 'pdf'):
        fig.savefig(figures / f'{stem}.{suffix}', bbox_inches='tight')


print('Results folder:', output, '· parallel workers:', N_JOBS)

### 1.1 · Inputs: two datasets

Notebook 66's input construction, copied unchanged (its cell 4):
- **Human vs mouse PT** is notebook 37's construction on SCF13 (`pseudospace.coordinate_inputs.pt_inputs`).
- **AKI vs control mouse PT** mirrors notebook 45's AKI positive control (its cells 4, 8 and 10).

In both, the library size is the total count over all orthologs measured in the dataset, and the NB
offset is its log. Two per-gene quantities are added for this notebook:
- the squeezeVar covariate, the log of the gene's mean raw count per structure over all structures
  of the dataset;
- a flag for genes with zero counts in at least one specimen. They stay in the primary and are
  dropped in one sensitivity.

In [ ]:
import anndata as ad
from scipy import sparse

from pseudospace.coordinate_inputs import pt_inputs, read_coordinate
from pseudospace.pathway_pipelines import partition_groups
from pseudospace.pathway_remodeling import matching_strata
from pseudospace.pathways import build_pathway_membership
from pseudospace.stage_cache import cached_payload, digest

t0 = time.perf_counter()
hm = pt_inputs(results_root, data_root, read_coordinate(coordinate_file))
assert len(hm['position']) == 12272 and len(hm['genes']) == 11071 and len(hm['gene_sets']) == 1513, "notebook 37's universe changed"
DATA = {'human_vs_mouse': {
    'title': 'Human vs mouse PT', 'case_name': 'human', 'reference_name': 'mouse', 'coordinate': 'SCF13',
    'counts': hm['counts'].tocsr(), 'library': hm['library'], 'position': hm['position'],
    'specimen': hm['specimen'].astype(str), 'segment': pd.Series(hm['segment']).str.replace('PT-', '', regex=False).to_numpy(),
    'case': hm['human'].astype(bool), 'genes': list(hm['genes']), 'sets': hm['gene_sets'], 'strata': hm['strata'],
    'lognorm': hm['Y'].tocsr()}}
del hm

# AKI vs control: notebook 45 cells 4, 8 and 10 (build_pool for one source of four mice), as notebook 66
universe_genes = pd.read_csv(universe_file, index_col=0).index.tolist()
orthologs = pd.read_csv(upstream13 / 'ortholog_map_used.csv')
coverage = pd.concat([build_pathway_membership(
    json.loads((data_root / 'mouse_vs_human' / 'pathway_gene_sets' / f'{lib}.json').read_text()), universe_genes,
    ortholog_map=orthologs, library_name=lib, tested=universe_genes, min_genes=1) for lib in LIBRARIES], ignore_index=True)
coverage['pathway_id'] = coverage.library + '::' + coverage.pathway
coverage = coverage[coverage.n_tested.between(SET_SIZES[0], min(SET_SIZES[1], len(universe_genes) - 1))]
gene_sets_all = {row.pathway_id: list(row.genes_present) for row in coverage.itertuples()}
assert len(gene_sets_all) == 1513, "notebook 37's pathways changed"

v5 = ad.read_h5ad(mouse_v5 / 'all_mouse_tubules_scanpy_dpt.h5ad')
v5 = v5[v5.obs.broad_tubule_marker_call.astype(str).eq('PT').to_numpy()
        & v5.obs.segment_class.astype(str).isin(['PT-S1', 'PT-S2', 'PT-S3']).to_numpy()].copy()
support = v5.obs.groupby('sample', observed=True).pt_subset_scanpy_dpt.quantile([.01, .99]).unstack()
coordinate = v5.obs.pt_subset_scanpy_dpt.to_numpy(float)
v5 = v5[(coordinate >= support[.01].max()) & (coordinate <= support[.99].min())].copy()
aki_position = v5.obs.pt_subset_scanpy_dpt.to_numpy(float)
aki_position = (aki_position - aki_position.min()) / (aki_position.max() - aki_position.min())
panel = np.isin(v5.var_names, list(set(orthologs.mouse_symbol)))
aki_raw = sparse.csr_matrix(v5.layers['counts'])[:, panel]
aki_var = v5.var_names[panel]
aki_specimen = v5.obs['sample'].astype(str).to_numpy()
aki_segment = v5.obs.segment_class.astype(str).str.replace('PT-', '', regex=False).to_numpy()
del v5
measured = sorted(set(aki_var[np.asarray(aki_raw.sum(axis=0)).ravel() > 0]))
aki_counts_all = sparse.csr_matrix(aki_raw[:, pd.Index(aki_var).get_indexer(measured)], dtype=float)
aki_library = np.asarray(aki_counts_all.sum(axis=1)).ravel()
aki_lognorm_all = aki_counts_all.multiply((1e4 / aki_library)[:, None]).tocsr()
aki_lognorm_all.data = np.log1p(aki_lognorm_all.data)
pool_genes = [g for g in universe_genes if g in set(measured)]
columns = pd.Index(measured).get_indexer(pool_genes)
x = aki_lognorm_all[:, columns]
per_donor = {}
for name in np.unique(aki_specimen):
    rows, seg = x[aki_specimen == name], aki_segment[aki_specimen == name]
    present = np.array([np.asarray((rows[seg == s] > 0).sum(axis=0)).ravel() > 0 for s in np.unique(seg)])
    per_donor[name] = (np.asarray(rows.mean(axis=0)).ravel(), np.asarray((rows > 0).mean(axis=0)).ravel(), present.mean(axis=0))
aki_covariates = pd.DataFrame({key: np.mean([per_donor[d][k] for d in per_donor], axis=0)
                               for k, key in enumerate(('mean_expression', 'detection', 'coverage'))}, index=pool_genes)
aki_genes = list(aki_covariates.index[aki_covariates.detection.ge(.02)])
keep = set(aki_genes)
aki_sets = {p: [g for g in v if g in keep] for p, v in gene_sets_all.items()}
aki_sets = {p: v for p, v in aki_sets.items() if SET_SIZES[0] <= len(v) <= min(SET_SIZES[1], len(keep) - 1)}
aki_columns = pd.Index(measured).get_indexer(aki_genes)
DATA['aki_vs_control'] = {
    'title': 'AKI vs control mouse PT', 'case_name': 'AKI', 'reference_name': 'control', 'coordinate': 'mouse-only PT DPT',
    'counts': aki_counts_all[:, aki_columns].tocsr(), 'library': aki_library, 'position': aki_position,
    'specimen': aki_specimen, 'segment': aki_segment, 'case': np.isin(aki_specimen, ['IR2A2', 'IR2A4']),
    'genes': aki_genes, 'sets': aki_sets, 'strata': matching_strata(aki_covariates.loc[aki_genes], bins=3),
    'lognorm': aki_lognorm_all[:, aki_columns].tocsr()}
del aki_raw, aki_counts_all, aki_lognorm_all, x
assert len(aki_position) == 16991 and len(aki_genes) == 9107 and len(aki_sets) == 1429, "notebook 45's AKI inputs changed"
pools45 = sorted(nb45_cache.glob('pool_aki__*.npz'), key=lambda p: json.loads(Path(str(p) + '.key.json').read_text())['written_utc'])
if pools45:
    with np.load(pools45[-1]) as stored:
        assert list(stored['genes'].astype(str)) == aki_genes and list(stored['pathways'].astype(str)) == list(aki_sets)
        assert np.array_equal(stored['strata'], DATA['aki_vs_control']['strata'])
    print("AKI genes, pathways and strata = notebook 45's cached pool")

assert list(DATA) == DATASETS
PARTITIONS, SHORT = {}, {}


def group_one(ds, label):
    """The specimens in group 1 of a partition (the case specimens for the condition split)."""
    d = DATA[ds]
    return sorted(set(d['specimen'][PARTITIONS[ds][label][0] == 1]))


for ds, d in DATA.items():
    assert d['counts'].shape == d['lognorm'].shape == (len(d['position']), len(d['genes']))
    assert all(set(v) <= set(d['genes']) for v in d['sets'].values())
    PARTITIONS[ds] = partition_groups(d['specimen'], d['case'])
    labels = list(PARTITIONS[ds])
    SHORT[ds] = dict(zip(labels, ['condition', 'relabeling 1', 'relabeling 2']))
    detected = {side: np.asarray((d['counts'][mask] > 0).mean(axis=0)).ravel()
                for side, mask in (('case', d['case']), ('reference', ~d['case']))}
    d['detection_case'], d['detection_reference'] = detected['case'], detected['reference']
    d['nearly_absent'] = np.minimum(detected['case'], detected['reference']) < ABSENT
    d['mean_expression'] = np.mean([np.asarray(d['lognorm'][d['specimen'] == s].mean(axis=0)).ravel()
                                    for s in sorted(set(d['specimen']))], axis=0)     # equal-specimen mean log1p CP10k
    d['input_key'] = {'counts': digest(d['counts']), 'library': d['library'], 'position': d['position'],
                      'specimen': d['specimen'], 'genes': d['genes']}
    # This notebook: the squeezeVar covariate and the zero-count flag
    d['log_mean_count'] = np.log(np.asarray(d['counts'].mean(axis=0)).ravel())
    totals = np.array([np.asarray(d['counts'][d['specimen'] == s].sum(axis=0)).ravel() for s in sorted(set(d['specimen']))])
    d['zero_in_a_specimen'] = (totals == 0).any(axis=0)
    d['case_specimens'] = sorted(set(d['specimen'][d['case']]))
    d['reference_specimens'] = sorted(set(d['specimen'][~d['case']]))

INPUT_TABLE = pd.DataFrame({d['title']: {
    'structures': len(d['position']), 'genes': len(d['genes']), 'pathways': len(d['sets']),
    'reference specimens': ', '.join(f"{s} ({int((d['specimen'] == s).sum()):,})" for s in d['reference_specimens']),
    'case specimens': ', '.join(f"{s} ({int((d['specimen'] == s).sum()):,})" for s in d['case_specimens']),
    **{f'{SHORT[ds][label]} (one side)': ' + '.join(group_one(ds, label)) for label in list(PARTITIONS[ds])[1:]},
    'genes nearly absent in one group': f"{int(d['nearly_absent'].sum()):,} ({d['nearly_absent'].mean():.1%})",
    'genes with zero counts in a specimen': f"{int(d['zero_in_a_specimen'].sum()):,} ({d['zero_in_a_specimen'].mean():.1%})"}
    for ds, d in DATA.items()})
INPUT_TABLE.to_csv(tables / 'inputs.csv')
display(INPUT_TABLE)
print(f'Inputs {time.perf_counter() - t0:.0f} s')

### 1.2 · Redundancy clusters

Overlaps (section 7) are also counted over **redundancy clusters**, so that near-duplicate Reactome
pathways do not inflate them. The clusters are notebook 66's, read from its tables. Its rule was fixed
there before any result: 1 − Jaccard of membership within the tested genes, average linkage, cut at
mean Jaccard 0.5, each cluster named by its medoid (Caleb Hallinan's `cluster_gene_sets`).

In [ ]:
REDUNDANCY, NAMES = {}, {}
for ds, d in DATA.items():
    clusters = pd.read_csv(nb66 / 'tables' / f'redundancy_clusters_{ds}.csv', index_col='pathway_id')
    assert sorted(clusters.index) == sorted(d['sets']), "notebook 66's redundancy clusters cover other pathways"
    REDUNDANCY[ds] = clusters.reindex(list(d['sets']))
    NAMES[ds] = pd.Series({p: p.split('::', 1)[1] for p in d['sets']}).str.replace(r' R-HSA-\d+$', '', regex=True)
print({DATA[ds]['title']: f"{len(DATA[ds]['sets']):,} pathways in {REDUNDANCY[ds].cluster_id.nunique():,} redundancy clusters"
       for ds in DATASETS})


def representatives(ds, calls):
    """The redundancy clusters a call set touches, named by their medoids."""
    return set(REDUNDANCY[ds].representative.reindex(sorted(calls)))

## 2 · The numerator and the specimen-shape denominator

**The numerator** is `T_spatial` from the shared NB gene model (`standard_gene_model`), refitted here
exactly as in notebook 66: an NB GLM on raw counts, offset log(library), a 6-df cubic B-spline, prior
weights wᵢ = n / (2 · J · n_j), and one α per gene from the condition split's M_full, reused for both
relabelings. In a relabeling the condition's own smooth shape enters every model as a nuisance term.

**The denominator** (`pathway_pipelines.specimen_shape_lr`) uses the same α, weights and offset:
- M_full's columns are [intercept, basis, group, group × basis, within-group specimen intercepts,
  nuisance × basis];
- M_spec adds each group's within-group specimen contrasts × basis, so every specimen has its own
  intercept and shape;
- LR_spec = D(M_full) − D(M_spec), truncated at 0 like every LR here.

**Why df₂ is 12 and 6.** With 2 + 2 specimens, the four specimen shapes span four directions:
the common shape, the condition contrast and the two relabeling contrasts. The relabeling contrasts
are the two ways of pairing a case with a reference specimen.
- **Condition split.** M_full holds the common shape and the condition contrast. M_spec frees the
  other two contrasts: 2 × 6 = 12 columns.
- **Relabeling.** M_full already holds the species (condition) shape as nuisance, so M_spec frees
  only one contrast, 6 columns: the shape of the *other* relabeling.

So the condition split is judged against the specimen shapes of both relabelings, and each relabeling
against the other's. This follows from the protocol; it was not chosen here. The guard below checks
df₂ from the design ranks.

**Failures** follow nb_gam's conventions:
- LR_spec of a gene whose α, M_full or M_spec fit did not converge is replaced by the median of the
  converged genes. `T_spatial` is filled exactly as in notebook 66.
- Separated genes, whose design loses rank on the structures with a count (for example a specimen
  without counts), are flagged and never filled.

**Guards** (not cached):
- df₂ = 12 for the condition split and 6 for each relabeling, from the design ranks;
- `T_spatial`, its flags and α equal notebook 66's cached gene models for all three partitions of
  both datasets (rtol 1e-6);
- for human vs mouse, `T_spatial` equals notebook 65's NB statistics;
- the specimen curves are identical in all three partitions.

In [ ]:
from pseudospace import levelshape, nb_gam, pathway_calibration, pathway_pipelines, stats_gam
from pseudospace.pathway_pipelines import GeneModel, specimen_shape_lr, standard_gene_model

cache = output / 'stage_cache'
code_key = digest([NOTEBOOK_LOGIC_VERSION] + [Path(m.__file__) for m in
                                               (levelshape, nb_gam, pathway_calibration, pathway_pipelines, stats_gam)])
SECONDS = {}   # uncached compute time of every cached stage


def cached(stage, compute, *, params=None, inputs=None, key=None):
    """cached_payload under this notebook's root and code; keeps the stage's uncached compute time."""
    def wrapped():
        t = time.perf_counter()
        out = dict(compute())
        out['__seconds'] = np.array(time.perf_counter() - t)
        return out
    result = cached_payload(stage, wrapped, root=cache, params=params, inputs=inputs, code=key or code_key)
    SECONDS[stage] = float(result.pop('__seconds'))
    return result


def slug(label):
    return label.replace(':', '_').replace('+', '_')


def model_payload(model):
    out = {f'stat|{c}': model.statistics[c].to_numpy() for c in model.statistics}
    out.update(grid=model.grid, knots=model.knots, reference=model.reference, case=model.case, delta=model.delta,
               se=model.se, group_names=model.groups.index.to_numpy().astype(str), group_values=model.groups.to_numpy(),
               alpha=np.asarray(model.dispersion['alpha'], float), alpha_converged=np.asarray(model.dispersion['converged'], bool))
    for name, curve in model.specimen_curves.items():
        out[f'curve|{name}'] = curve
        out[f'converged|{name}'] = model.specimen_converged[name].to_numpy()
        out[f'separated|{name}'] = model.specimen_separated[name].to_numpy()
    return out


def model_from(payload, genes):
    index = pd.Index(genes, name='gene')
    stats = pd.DataFrame({c: payload[f'stat|{c}'] for c in ('T_level', 'T_spatial', 'T_total', 'alpha', 'converged',
                                                            'separated')}, index=index)
    stats[['converged', 'separated']] = stats[['converged', 'separated']].astype(bool)
    names = sorted(k.split('|', 1)[1] for k in payload if k.startswith('curve|'))
    return GeneModel(statistics=stats, grid=payload['grid'], knots=payload['knots'], reference=payload['reference'],
                     case=payload['case'], delta=payload['delta'], se=payload['se'],
                     specimen_curves={n: payload[f'curve|{n}'] for n in names},
                     specimen_converged=pd.DataFrame({n: payload[f'converged|{n}'].astype(bool) for n in names}, index=index),
                     specimen_separated=pd.DataFrame({n: payload[f'separated|{n}'].astype(bool) for n in names}, index=index),
                     groups=pd.Series(payload['group_values'].astype(int), index=payload['group_names'].astype(str)),
                     dispersion={'alpha': payload['alpha'], 'converged': payload['alpha_converged'].astype(bool)})


MODELS, DENOM = {}, {}
for ds, d in DATA.items():
    MODELS[ds], DENOM[ds] = {}, {}
    for label, (group, nuisance) in PARTITIONS[ds].items():
        dispersion = None if label == 'species' else MODELS[ds]['species'].dispersion
        payload = cached(f'gene_model_{ds}_{slug(label)}', lambda d=d, group=group, nuisance=nuisance, dispersion=dispersion:
                         model_payload(standard_gene_model(d['counts'], d['library'], d['position'], group, d['specimen'],
                                                           genes=d['genes'], nuisance_shape=nuisance, dispersion=dispersion,
                                                           basis_df=BASIS_DF, n_jobs=N_JOBS)),
                         params={'basis_df': BASIS_DF, 'grid_points': GRID_POINTS, 'label': label},
                         inputs={**d['input_key'], 'group': group, 'nuisance': nuisance,
                                 'alpha': None if dispersion is None else dispersion['alpha']})
        MODELS[ds][label] = model_from(payload, d['genes'])
        alpha = MODELS[ds]['species'].dispersion
        DENOM[ds][label] = cached(f'specimen_shape_{ds}_{slug(label)}',
                                  lambda d=d, group=group, nuisance=nuisance, alpha=alpha:
                                  specimen_shape_lr(d['counts'], d['library'], d['position'], group, d['specimen'], alpha,
                                                    nuisance_shape=nuisance, basis_df=BASIS_DF, n_jobs=N_JOBS),
                                  params={'basis_df': BASIS_DF, 'label': label},
                                  inputs={**d['input_key'], 'group': group, 'nuisance': nuisance, 'alpha': alpha['alpha'],
                                          'alpha_converged': alpha['converged']})
    d['log_ratio'] = pd.Series(MODELS[ds]['species'].delta.mean(axis=1), index=d['genes'])   # mean NB log rate ratio


def stored66(stage):
    """Notebook 66's newest cached payload of one stage (read only), or None when its cache was purged."""
    files = sorted((nb66 / 'stage_cache').glob(f'{stage}__*.npz'),
                   key=lambda p: json.loads(Path(str(p) + '.key.json').read_text())['written_utc'])
    if not files:
        return None
    with np.load(files[-1], allow_pickle=False) as stored:
        return {k: stored[k] for k in stored.files}


guard_log, checked66 = [], []
for ds, d in DATA.items():
    condition = MODELS[ds]['species']
    assert len(condition.grid) == GRID_POINTS
    for label, model in MODELS[ds].items():
        expected = DF2['condition' if label == 'species' else 'relabeling']
        assert int(DENOM[ds][label]['df2']) == expected, (ds, label, int(DENOM[ds][label]['df2']))
        np.testing.assert_array_equal(model.grid, condition.grid)
        for name, curve in model.specimen_curves.items():
            np.testing.assert_allclose(curve, condition.specimen_curves[name], rtol=1e-9, atol=1e-9)
        old = stored66(f'gene_model_{ds}_{slug(label)}')
        if old is not None:
            np.testing.assert_allclose(model.statistics.T_spatial, old['stat|T_spatial'], rtol=1e-6, atol=1e-6)
            np.testing.assert_array_equal(model.statistics.converged, old['stat|converged'].astype(bool))
            np.testing.assert_array_equal(model.statistics.separated, old['stat|separated'].astype(bool))
            np.testing.assert_allclose(model.dispersion['alpha'], old['alpha'], rtol=1e-6)
            checked66.append(f'{ds} {SHORT[ds][label]}')
guard_log.append(f"df2 = {DF2['condition']} (condition split) and {DF2['relabeling']} (each relabeling) from the design "
                 'ranks, both datasets')
guard_log.append(f"T_spatial, its converged/separated flags and α = notebook 66's cached gene models (rtol 1e-6): "
                 f"{len(checked66)} of 6 partitions checked ({', '.join(checked66) or 'cache not found'})")
if nb65_genes.is_file():
    nb65 = pd.read_csv(nb65_genes, index_col=0).reindex(DATA['human_vs_mouse']['genes'])
    if 'T_spatial_nb' in nb65:
        for label, model in MODELS['human_vs_mouse'].items():
            column = 'T_spatial_nb' if label == 'species' else f'T_spatial_nb|{label}'
            np.testing.assert_allclose(model.statistics.T_spatial, nb65[column], rtol=1e-6, atol=1e-6)
        guard_log.append("human vs mouse: T_spatial of all three partitions = notebook 65's NB statistics (rtol 1e-6)")
guard_log.append('specimen curves identical in all three partitions of both datasets')
print('\n'.join('· ' + line for line in guard_log))

DENOM_ROWS = []
for ds in DATASETS:
    for label, model in MODELS[ds].items():
        den = DENOM[ds][label]
        df2 = int(den['df2'])
        s2 = np.asarray(den['LR_spec'], float) / df2
        DENOM_ROWS.append({
            'dataset': DATA[ds]['title'], 'partition': SHORT[ds][label], 'df1': DF1, 'df2': df2,
            'residual df': int(den['residual_df']),
            'LR_spec non-converged (median-filled)': int((~np.asarray(den['converged'], bool)).sum()),
            'separated in M_full or M_spec (flagged)': int(np.asarray(den['separated'], bool).sum()),
            'LR_spec = 0': int((s2 == 0).sum()), 'median T_spatial / df1': float((model.statistics.T_spatial / DF1).median()),
            'median LR_spec / df2': float(np.median(s2)),
            'median unmoderated F': float(np.median((model.statistics.T_spatial.to_numpy() / DF1) / np.maximum(s2, 1e-12))),
            'gene model seconds': round(SECONDS[f'gene_model_{ds}_{slug(label)}'], 1),
            'denominator seconds': round(SECONDS[f'specimen_shape_{ds}_{slug(label)}'], 1)})
DENOM_TABLE = pd.DataFrame(DENOM_ROWS)
DENOM_TABLE.to_csv(tables / 'numerator_and_denominator.csv', index=False)
display(DENOM_TABLE.round(3))

## 3 · The moderated F

For each partition, s² = LR_spec / df₂ is moderated across genes by `limma::squeezeVar`
(`pathway_pipelines.squeeze_var`, through rpy2; limma's defaults otherwise). With one df for all
genes, limma uses its legacy `fitFDist`, or `fitFDistRobustly` when `robust = TRUE`.

- **Primary:** the covariate is the log mean count per structure. limma fits a natural-spline trend
  of the prior variance s₀² on it, so the prior follows the gene's expression.
- **Sensitivities:** no covariate; `robust = TRUE` (with the primary's covariate); and the primary
  rerun without the genes that have zero counts in any specimen (squeezeVar refitted on the remaining
  genes).

Then s²_post = (df₂ s² + d₀ s₀²) / (df₂ + d₀) and F = (`T_spatial` / 6) / s²_post with
df (6, df₂ + d₀). Where limma's prior df d₀ is infinite, s²_post = s₀² and the reference distribution
is χ²₆ / 6, as limma treats it.

The p-value is computed in log space (`pathway_pipelines.log_f_sf`), accurately in both tails.
z = Φ⁻¹(1 − p) is obtained from log p through `scipy.special.ndtri_exp`, so z stays finite both for
tiny p and for p within 10⁻²⁰ of 1. A
gene with F = 0 (`T_spatial` truncated at 0) would get z = −∞. It is tied with the smallest finite z
(section 10).

**What it can tell you:** whether a gene's group shape difference is large relative to how much its
specimens differ in shape within a group. **What it cannot tell you:** with two specimens per group,
each gene's own s² rests on few degrees of freedom. The prior's share of s²_post is
d₀ / (df₂ + d₀); when it is large, the test is mostly a ratio to the expression-dependent prior.

In [ ]:
from scipy.special import ndtri_exp
from scipy.stats import false_discovery_control

from pseudospace.pathway_pipelines import log_f_sf, moderated_f, rank_normal_scores

VARIANTS = {'primary': {'covariate': True, 'robust': False}, 'no covariate': {'covariate': False, 'robust': False},
            'robust': {'covariate': True, 'robust': True}, 'no zero-count genes': {'covariate': True, 'robust': False}}
STAT = {ds: {v: {} for v in VARIANTS} for ds in DATASETS}
SQUEEZE_ROWS = []
for ds, d in DATA.items():
    for label, model in MODELS[ds].items():
        den = DENOM[ds][label]
        df2 = int(den['df2'])
        converged = model.statistics.converged.to_numpy() & np.asarray(den['converged'], bool)
        separated = model.statistics.separated.to_numpy() | np.asarray(den['separated'], bool)
        for variant, options in VARIANTS.items():
            keep = ~d['zero_in_a_specimen'] if variant == 'no zero-count genes' else np.ones(len(d['genes']), bool)
            frame = moderated_f(model.statistics.T_spatial[keep], DF1, np.asarray(den['LR_spec'], float)[keep], df2,
                                covariate=d['log_mean_count'][keep] if options['covariate'] else None,
                                robust=options['robust'])
            frame['q'] = false_discovery_control(frame.p.to_numpy())
            frame['converged'], frame['separated'] = converged[keep], separated[keep]
            assert np.isfinite(frame.z).all() and np.isfinite(frame.F).all()
            STAT[ds][variant][label] = frame
            d0 = frame.d0.to_numpy()
            SQUEEZE_ROWS.append({
                'dataset': d['title'], 'partition': SHORT[ds][label], 'variant': variant, 'genes': int(keep.sum()), 'df2': df2,
                'd0 (median)': float(np.median(d0)), 'share d0 = inf': float(np.isinf(d0).mean()),
                's0² (median)': float(frame.s2_prior.median()),
                's0² range (5–95%)': f'{frame.s2_prior.quantile(.05):.3g}–{frame.s2_prior.quantile(.95):.3g}',
                'median s²_post / s²': float(np.median(frame.s2_post / np.maximum(frame.s2, 1e-12))),
                'median F': float(frame.F.median()), 'genes F = 0': int((frame.F == 0).sum()),
                'genes p ≤ 0.05': int((frame.p <= .05).sum()), 'genes BH ≤ 0.05': int((frame.q <= GENE_FDR).sum())})
SQUEEZE = pd.DataFrame(SQUEEZE_ROWS)
SQUEEZE.to_csv(tables / 'moderation.csv', index=False)
display(SQUEEZE.set_index(['dataset', 'partition', 'variant']).round(3))
for ds in DATASETS:
    for variant in VARIANTS:
        pd.concat({SHORT[ds][label]: STAT[ds][variant][label] for label in PARTITIONS[ds]}, axis=1).to_csv(
            tables / f"gene_statistics_{ds}_{variant.replace(' ', '_').replace('-', '_')}.csv")

**The figure.** For each partition, every gene's s² = LR_spec / df₂ (grey dots) against the
covariate. The line is the primary's prior s₀² (squeezeVar with the covariate). The dashed line is
the prior without the covariate. Each panel gives d₀: how many degrees of freedom of prior
information each gene's s² is shrunk with.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(180 * MM, 92 * MM), sharex='row', sharey='row', gridspec_kw={'hspace': .55, 'wspace': .12})
for row, ds in zip(axes, DATASETS):
    d = DATA[ds]
    order = np.argsort(d['log_mean_count'])
    for ax, label in zip(row, PARTITIONS[ds]):
        primary, flat = STAT[ds]['primary'][label], STAT[ds]['no covariate'][label]
        s2 = np.maximum(primary.s2.to_numpy(), 1e-4)
        ax.scatter(d['log_mean_count'], s2, s=.6, color=REF_COLOR, alpha=.25, linewidth=0, rasterized=True)
        ax.plot(d['log_mean_count'][order], primary.s2_prior.to_numpy()[order], color=N_COLOR, lw=1.1)
        ax.axhline(flat.s2_prior.iloc[0], color=INK, lw=.7, ls='--')
        ax.set_yscale('log')
        d0 = primary.d0.iloc[0]
        ax.set_title(f"{d['title']} · {SHORT[ds][label]}", loc='left')
        ax.text(.98, .96, f"df₂ = {int(DENOM[ds][label]['df2'])}; d₀ = {d0:.1f} (with covariate), "
                f"{flat.d0.iloc[0]:.1f} (without)", transform=ax.transAxes, ha='right', va='top', fontsize=5.2, color=INK)
        ax.set_xlabel('log mean count per structure')
    row[0].set_ylabel('s² = LR_spec / df₂ (floored at 10⁻⁴)')
handles = [plt.Line2D([], [], color=N_COLOR, lw=1.1, label='prior s₀², covariate (primary)'),
           plt.Line2D([], [], color=INK, lw=.7, ls='--', label='prior s₀², no covariate'),
           plt.Line2D([], [], marker='o', lw=0, color=REF_COLOR, markersize=3, label='genes')]
fig.legend(handles=handles, loc='upper center', ncol=3, bbox_to_anchor=(.5, .0))
save(fig, 'fig1_squeezevar_prior')
plt.show()

## 4 · Pathway tests on z

Every test runs on the condition split and both relabelings, on the gene sets of notebook 66
(10–300 tested genes):

| Key | Test | Input | A call |
|---|---|---|---|
| cameraPR (**primary**) | `limma::cameraPR`, `inter.gene.cor` 0.01, `use.ranks` FALSE | z of the primary F | Up, FDR ≤ 0.05 |
| GSEA | gseapy prerank, 1,000 gene-set permutations, seed 12 (`rank_first_gsea`, metric raw) | z of the primary F | NES > 0, FDR ≤ 0.05 |
| ORA | `gseapy.enrich` (hypergeometric), background = all tested genes (`over_representation`) | genes with BH ≤ 0.05 on the primary F | FDR ≤ 0.05 |
| cameraPR rank-normal F | as the primary | rank-normal scores of the primary F | as the primary |
| cameraPR no covariate | as the primary | z with squeezeVar without the covariate | as the primary |
| cameraPR robust | as the primary | z with squeezeVar `robust = TRUE` | as the primary |
| cameraPR no zero-count genes | as the primary | z refitted without the genes with zero counts in a specimen; sets re-filtered to 10–300 remaining genes | as the primary |

**The one-sided p for calibration:**
- cameraPR: p/2 when Up, else 1 − p/2;
- GSEA: its nominal p made one-sided by the sign of NES;
- ORA: the hypergeometric p of the single gene list. This is notebook 61/66's list-based convention
  (best list p × number of lists) with one list; a pathway without a listed gene gets p = 1.

In [ ]:
from pseudospace.pathway_pipelines import camera_pr, over_representation, rank_first_gsea

N_METHODS = {
    'cameraPR': 'N · cameraPR on z (primary)',
    'GSEA': 'N · GSEA prerank on z',
    'ORA': 'N · ORA on genes with BH ≤ 0.05',
    'cameraPR rank-normal F': 'N · cameraPR on rank-normal F (sensitivity)',
    'cameraPR no covariate': 'N · cameraPR on z, squeezeVar without covariate (sensitivity)',
    'cameraPR robust': 'N · cameraPR on z, squeezeVar robust = TRUE (sensitivity)',
    'cameraPR no zero-count genes': 'N · cameraPR on z, zero-count-specimen genes dropped (sensitivity)'}
TESTS = {ds: {m: {} for m in N_METHODS} for ds in DATASETS}
RUNTIME = {ds: {m: 0. for m in N_METHODS} for ds in DATASETS}
SETS_KEPT, ORA_LISTS = {}, {ds: {} for ds in DATASETS}


def timed(ds, method, label, run):
    t = time.perf_counter()
    TESTS[ds][method][label] = run()
    RUNTIME[ds][method] += time.perf_counter() - t


for ds, d in DATA.items():
    kept = set(np.asarray(d['genes'])[~d['zero_in_a_specimen']])
    sets = {p: [g for g in v if g in kept] for p, v in d['sets'].items()}
    SETS_KEPT[ds] = {p: v for p, v in sets.items() if SET_SIZES[0] <= len(v) <= min(SET_SIZES[1], len(kept) - 1)}
    for label in PARTITIONS[ds]:
        primary = STAT[ds]['primary'][label]
        camera = lambda score, sets=d['sets']: camera_pr(score, sets, inter_gene_cor=CAMERA_COR, use_ranks=False, fdr=ALPHA)  # noqa: E731
        timed(ds, 'cameraPR', label, lambda: camera(primary.z))
        timed(ds, 'GSEA', label, lambda: rank_first_gsea(primary.z, d['sets'], metric='raw', min_size=SET_SIZES[0],
                                                          max_size=SET_SIZES[1], permutation_num=GSEA_PERMUTATIONS,
                                                          seed=GSEA_SEED, threads=min(N_JOBS, 32), fdr=ALPHA))
        listed = list(primary.index[primary.q.le(GENE_FDR)])
        ORA_LISTS[ds][label] = listed
        timed(ds, 'ORA', label, lambda: over_representation(listed, d['sets'], d['genes'], fdr=ALPHA).assign(
            p_one_sided=lambda t: t.p))
        timed(ds, 'cameraPR rank-normal F', label, lambda: camera(rank_normal_scores(primary.F)))
        timed(ds, 'cameraPR no covariate', label, lambda: camera(STAT[ds]['no covariate'][label].z))
        timed(ds, 'cameraPR robust', label, lambda: camera(STAT[ds]['robust'][label].z))
        timed(ds, 'cameraPR no zero-count genes', label,
              lambda: camera(STAT[ds]['no zero-count genes'][label].z, SETS_KEPT[ds]))
    for method, runs in TESTS[ds].items():
        for label, table in runs.items():
            expected = SETS_KEPT[ds] if method == 'cameraPR no zero-count genes' else d['sets']
            assert list(table.pathway_id) == list(expected), (ds, method, label)
            table.assign(partition=label).to_csv(tables / f"pathway_tests_{ds}_{method.replace(' ', '_').replace('-', '_')}_{slug(label)}.csv",
                                                 index=False)

N_CALLS = {ds: {m: {label: set(TESTS[ds][m][label].pathway_id[TESTS[ds][m][label].call.astype(bool)])
                    for label in PARTITIONS[ds]} for m in N_METHODS} for ds in DATASETS}
print('ORA gene lists (genes with BH ≤ 0.05):', {DATA[ds]['title']: {SHORT[ds][p]: len(v) for p, v in ORA_LISTS[ds].items()}
                                                 for ds in DATASETS})
display(pd.DataFrame({(DATA[ds]['title'], SHORT[ds][label]): {N_METHODS[m]: len(N_CALLS[ds][m][label]) for m in N_METHODS}
                      for ds in DATASETS for label in PARTITIONS[ds]}))

## 5 · The method tables

The columns are notebook 66's:
- **calls:** pathways called for the condition split; the next column gives the redundancy clusters
  they touch;
- **relabeled calls:** calls under the two relabelings;
- **NCDR:** mean relabeled calls ÷ condition calls;
- **relabeling rule** (notebook 31): NCDR < 0.25 and each relabeling below 5% of the tested
  pathways;
- **calibration share:** of the 2 × (tested pathways) pooled relabeled one-sided p-values, the share
  ≤ 0.05. A calibrated test gives about 0.05; the limit is 0.075 (notebook 61's criterion 2);
- **seconds:** compute time of the pathway step for the three partitions.

The rows marked **66 ·** are read from notebook 66's `method_table.csv`, not recomputed:
- cameraPR and GSEA on rank-normal raw `T_spatial`;
- our NB joint test, whose VIF is fitted on the same relabelings;
- clustering-first ORA with the single-split null.

The primary **passes** on a dataset when it passes both the relabeling rule and the calibration
limit.

In [ ]:
RETURNS = {'cameraPR': 'per pathway: direction, p, FDR', 'GSEA': 'per pathway: NES, p, FDR, leading-edge genes',
           'ORA': 'per pathway: overlap with the gene list, p, FDR'}
COMPARE66 = {'A · cameraPR (primary)': '66 · cameraPR on rank-normal raw T_spatial',
             'A · GSEA, rank-normal scores': '66 · GSEA on rank-normal raw T_spatial',
             'Reference · NB joint test, relabeling-fitted VIF (ours)': '66 · NB joint test, relabeling-fitted VIF',
             'B · ORA, single-split null (like for like)': '66 · clustering-first ORA, single-split null'}
TABLE66 = pd.read_csv(nb66 / 'tables' / 'method_table.csv')
METHOD_ROWS, PLOT_ROWS = [], []
for ds in DATASETS:
    d = DATA[ds]
    condition, r1, r2 = list(PARTITIONS[ds])

    def add(key, label, family, condition_sets, relabeled_sets, p_relabeled, seconds, tested, note=''):
        n_condition = [len(c) for c in condition_sets]
        n_relabeled = [len(r) for r in relabeled_sets]
        mean_condition = float(np.mean(n_condition))
        ncdr = np.mean(n_relabeled) / mean_condition if mean_condition else np.nan
        share = float(np.mean(np.asarray(p_relabeled, float) <= ALPHA)) if p_relabeled is not None else np.nan
        METHOD_ROWS.append({
            'dataset': d['title'], 'method': label, 'family': family,
            'calls': ', '.join(map(str, n_condition)),
            'calls (redundancy clusters)': ', '.join(str(len(representatives(ds, c))) for c in condition_sets),
            'relabeled calls': ', '.join(map(str, n_relabeled)), 'NCDR': ncdr,
            'relabeling rule': 'pass' if mean_condition and ncdr < NCDR_MAX and max(n_relabeled) < NULL_RATE * tested else 'fail',
            'calibration share': share,
            'calibration': ('n/a' if p_relabeled is None else 'pass' if share <= CALIBRATION_MAX else 'fail') + note,
            'seconds': round(seconds, 1), 'returns': RETURNS[key]})
        PLOT_ROWS.append({'dataset': ds, 'method': label, 'family': family, 'condition': n_condition,
                          'relabeled': n_relabeled, 'tested': tested})

    for m, label in N_METHODS.items():
        tested = len(SETS_KEPT[ds]) if m == 'cameraPR no zero-count genes' else len(d['sets'])
        add(m.split(' ')[0], label, 'normalized', [N_CALLS[ds][m][condition]], [N_CALLS[ds][m][r1], N_CALLS[ds][m][r2]],
            np.concatenate([TESTS[ds][m][r].p_one_sided.to_numpy(float) for r in (r1, r2)]), RUNTIME[ds][m], tested,
            ' (list-based convention)' if m == 'ORA' else '')
    for original, label in COMPARE66.items():
        row = TABLE66[TABLE66.dataset.eq(d['title']) & TABLE66.method.eq(original)]
        assert len(row) == 1, (ds, original)
        row = row.iloc[0].to_dict()
        row.update(method=label, family='66 · ' + row['family'])
        METHOD_ROWS.append(row)
        PLOT_ROWS.append({'dataset': ds, 'method': label, 'family': row['family'],
                          'condition': [int(x) for x in str(row['calls']).split(', ')],
                          'relabeled': [int(x) for x in str(row['relabeled calls']).split(', ')], 'tested': len(d['sets'])})
METHOD_TABLE = pd.DataFrame(METHOD_ROWS)[list(TABLE66.columns)]
print('Pathways tested without the zero-count-specimen genes (10–300 remaining members):',
      {DATA[ds]['title']: f"{len(SETS_KEPT[ds]):,} of {len(DATA[ds]['sets']):,}" for ds in DATASETS})
METHOD_TABLE.to_csv(tables / 'method_table.csv', index=False)
PASS = {}
for ds in DATASETS:
    rows = METHOD_TABLE[METHOD_TABLE.dataset.eq(DATA[ds]['title'])].set_index('method')
    primary = rows.loc[N_METHODS['cameraPR']]
    PASS[ds] = primary['relabeling rule'] == 'pass' and primary['calibration'].startswith('pass')
    print(f"{DATA[ds]['title']}: {len(DATA[ds]['sets']):,} pathways tested")
    with pd.option_context('display.max_colwidth', 70):
        display(rows.drop(columns='dataset').round(3))
for ds in DATASETS:
    row = METHOD_TABLE[METHOD_TABLE.dataset.eq(DATA[ds]['title']) & METHOD_TABLE.method.eq(N_METHODS['cameraPR'])].iloc[0]
    print(f"{DATA[ds]['title']}: the primary {'PASSES' if PASS[ds] else 'FAILS'} — {row['calls']} calls; relabeled "
          f"{row['relabeled calls']}; NCDR {row['NCDR']:.2f}; calibration share {row['calibration share']:.3f} "
          f"(relabeling rule {row['relabeling rule']}, calibration {row['calibration']})")

**The figure.** One row per method. Filled dots: the condition split's calls (two dots for the
single-split row, one per null). Open dots: the two relabelings. The dashed line marks 5% of the
tested pathways, the most one relabeling may call. A specific method has its filled dot far to the
right of its open dots. Right margin: NCDR and the calibration share.

In [ ]:
FAMILY_COLOR = {'normalized': N_COLOR, '66 · A': A_COLOR, '66 · B': B_COLOR, '66 · reference': REF_COLOR}
fig, axes = plt.subplots(1, 2, figsize=(180 * MM, 74 * MM), sharey=True, gridspec_kw={'wspace': .62})
for ax, ds in zip(axes, DATASETS):
    rows = {r['method']: r for r in PLOT_ROWS if r['dataset'] == ds}
    order = list(rows)
    table = METHOD_TABLE[METHOD_TABLE.dataset.eq(DATA[ds]['title'])].set_index('method')
    for i, method in enumerate(order):
        y, r = len(order) - 1 - i, rows[method]
        color = FAMILY_COLOR[r['family']]
        ax.plot([0, max(r['condition'] + r['relabeled'])], [y, y], color=LIGHT, lw=.6, zorder=1)
        ax.scatter(r['relabeled'], [y] * 2, s=22, facecolor='white', edgecolor=color, linewidth=1, zorder=3)
        ax.scatter(r['condition'], [y] * len(r['condition']), s=22, color=color, edgecolor='white', linewidth=.5, zorder=4)
        row = table.loc[method]
        cal = '—' if pd.isna(row['calibration share']) else f"{row['calibration share']:.3f}"
        flag = [] if row['relabeling rule'] == 'pass' else ['fails relabeling']
        flag += ['fails calibration'] if str(row['calibration']).startswith('fail') else []
        ncdr = '—' if pd.isna(row['NCDR']) else f"{row['NCDR']:.2f}"
        ax.text(1.02, y, f"NCDR {ncdr} · cal. {cal}" + (f"\n{', '.join(flag)}" if flag else ''),
                transform=ax.get_yaxis_transform(), va='center', fontsize=5.2, color=INK, linespacing=1.1)
    ax.axvline(NULL_RATE * len(DATA[ds]['sets']), color=MUTED, lw=.6, ls='--', zorder=0)
    ax.text(NULL_RATE * len(DATA[ds]['sets']), -.75, ' 5% of tested', fontsize=5, color=MUTED, va='center')
    ax.set_xlim(left=-5)
    ax.set_ylim(-1.1, len(order) - .5)
    ax.set_xlabel('pathways called')
    ax.set_title(DATA[ds]['title'], loc='left')
labels = [r['method'] for r in PLOT_ROWS if r['dataset'] == DATASETS[0]]
axes[0].set_yticks(range(len(labels)), [m if len(m) < 72 else m[:70] + '…' for m in labels[::-1]])
for tick, method in zip(axes[0].get_yticklabels(), labels[::-1]):
    tick.set_color(FAMILY_COLOR[next(r['family'] for r in PLOT_ROWS if r['method'] == method)])
handles = [plt.Line2D([], [], marker='o', lw=0, color=INK, markersize=4, label='condition split'),
           plt.Line2D([], [], marker='o', lw=0, markerfacecolor='white', markeredgecolor=INK, markersize=4,
                      label='relabelings'),
           *[plt.Line2D([], [], marker='s', lw=0, color=c, markersize=4, label=f) for f, c in
             (('normalized (this notebook)', N_COLOR), ('66 · F ranking', A_COLOR), ('66 · our joint test', REF_COLOR),
              ('66 · clustering-first', B_COLOR))]]
fig.legend(handles=handles, loc='upper center', ncol=6, bbox_to_anchor=(.5, -.02))
save(fig, 'fig2_method_tables')
plt.show()

## 6 · Gene-level information checks

Reported for information; nothing is selected on them. For each partition and statistic:
- **share p ≤ 0.01 and p ≤ 0.05** over all tested genes;
- **λ (median-based):** the χ²₁ quantile of the median p divided by the χ²₁ median (0.455), the
  genomic-control factor; 1 for uniform p;
- **Spearman of z with mean expression:** the equal-specimen mean log1p CP10k of notebook 66.

For raw `T_spatial` the p is its nominal χ²₆ p. The statistic is a rank by design, because its
degrees of freedom count structures, so this p is shown only for contrast. Under a relabeling a
calibrated gene statistic gives shares of about 0.01 and 0.05 and λ ≈ 1. The QQ plot shows the
relabelings' gene p-values for both statistics.

In [ ]:
from scipy.stats import chi2, spearmanr

LAMBDA_MEDIAN = chi2.ppf(.5, 1)
RAW = {ds: {} for ds in DATASETS}
CHECK_ROWS = []
for ds, d in DATA.items():
    expression = pd.Series(d['mean_expression'], index=d['genes'])
    for label, model in MODELS[ds].items():
        log_p = log_f_sf(model.statistics.T_spatial.to_numpy() / DF1, DF1, np.inf)      # nominal chi2_6 p of T_spatial
        RAW[ds][label] = pd.DataFrame({'p': np.exp(log_p), 'z': -ndtri_exp(log_p)}, index=model.statistics.index)
        for name, frame in (('raw T_spatial (nominal χ²₆ p)', RAW[ds][label]),
                            *[(f'moderated F, {v}', STAT[ds][v][label]) for v in VARIANTS]):
            p, z = frame.p.to_numpy(), np.clip(frame.z.to_numpy(), -40, 40)
            CHECK_ROWS.append({'dataset': d['title'], 'partition': SHORT[ds][label], 'statistic': name, 'genes': len(frame),
                               'share p ≤ 0.01': float(np.mean(p <= .01)), 'share p ≤ 0.05': float(np.mean(p <= .05)),
                               'λ (median)': float(chi2.isf(np.median(p), 1) / LAMBDA_MEDIAN),
                               'Spearman z vs mean expression': float(spearmanr(z, expression.reindex(frame.index))[0])})
GENE_CHECKS = pd.DataFrame(CHECK_ROWS)
GENE_CHECKS.to_csv(tables / 'gene_level_checks.csv', index=False)
shown = GENE_CHECKS[GENE_CHECKS.statistic.isin(['raw T_spatial (nominal χ²₆ p)', 'moderated F, primary'])]
display(shown.set_index(['dataset', 'partition', 'statistic']).round(3))
display(GENE_CHECKS[~GENE_CHECKS.index.isin(shown.index)].set_index(['dataset', 'partition', 'statistic']).round(3))

fig, axes = plt.subplots(1, 2, figsize=(120 * MM, 56 * MM), gridspec_kw={'wspace': .35})
for ax, ds in zip(axes, DATASETS):
    for label, ls in zip(list(PARTITIONS[ds])[1:], ('-', '--')):
        for frame, color, name in ((RAW[ds][label], REF_COLOR, 'raw T_spatial'), (STAT[ds]['primary'][label], N_COLOR, 'moderated F')):
            observed = np.sort(-np.log10(np.clip(frame.p.to_numpy(), 1e-300, 1)))[::-1]
            expected = -np.log10((np.arange(1, len(observed) + 1) - .5) / len(observed))
            ax.plot(expected, np.minimum(observed, 30), color=color, ls=ls, lw=.9, label=f'{name} · {SHORT[ds][label]}')
    top = -np.log10(.5 / len(DATA[ds]['genes']))
    ax.plot([0, top], [0, top], color=MUTED, lw=.6, ls=':')
    ax.set_xlabel('expected −log10 p (uniform)')
    ax.set_ylabel('observed −log10 p (capped at 30)')
    ax.set_title(f"{DATA[ds]['title']}: genes under relabeling", loc='left')
axes[1].legend(loc='upper left', fontsize=5.2)
save(fig, 'fig3_gene_qq_relabelings')
plt.show()

### 6.1 · Why the primary fails calibration on AKI (post hoc)

This subsection was added after the method tables showed that the primary passes on human vs mouse
but fails the calibration limit on AKI. It changes no call and no pass/fail result above.

**The observation.** Under the AKI relabelings the gene-level p-values of the moderated F are close
to calibrated (the table above: λ 0.96 and 1.00; share p ≤ 0.05 of 0.050 and 0.061). Yet 10% of the
relabeled pathway p-values are ≤ 0.05. If the genes are calibrated one by one, the excess must come
from dependence between a pathway's genes, beyond what cameraPR's `inter.gene.cor` = 0.01 allows.

**The check.** As in notebook 66 (section 5.2), the relabeled one-sided pathway p-values are turned
into z and fitted to E[z²] = a(1 + (m − 1)ρ) (`pathway_decoys.decoy_inflation`). This is done for the
primary and two of its sensitivities.
- **ρ** is inflation that grows with set size: a uniform extra correlation between genes.
- **a** is inflation of the same size for every set, as when a minority of pathways carry strongly
  coherent specimen differences.

Notebook 66's fit for cameraPR on raw `T_spatial` is shown beside it. The relabeled calls are listed,
each with whether the condition split calls it too.

In [ ]:
from scipy.stats import norm

from pseudospace.pathway_decoys import decoy_inflation

WHY66 = pd.read_csv(nb66 / 'tables' / 'why_method_a_is_not_specific.csv')
POSTHOC_ROWS, RELABELED_ROWS = [], []
for ds, d in DATA.items():
    condition, r1, r2 = list(PARTITIONS[ds])
    sizes = pd.Series({p: len(v) for p, v in d['sets'].items()})
    lam = GENE_CHECKS[GENE_CHECKS.dataset.eq(d['title']) & GENE_CHECKS.statistic.eq('moderated F, primary')
                      & GENE_CHECKS.partition.ne('condition')]['λ (median)']
    for key in ('cameraPR', 'cameraPR rank-normal F', 'cameraPR no covariate'):
        tests = TESTS[ds][key]
        z = pd.DataFrame({r: tests[r].set_index('pathway_id').p_one_sided.clip(1e-300, 1 - 1e-16).map(norm.isf) for r in (r1, r2)})
        a, rho, vif = decoy_inflation(z, sizes)
        POSTHOC_ROWS.append({'dataset': d['title'], 'test': N_METHODS[key], 'implied a': a, 'implied ρ': rho,
                             'implied median VIF': float(vif.median()),
                             'gene-level λ under relabeling': ', '.join(f'{v:.2f}' for v in lam)})
    row66 = WHY66[WHY66.dataset.eq(d['title']) & WHY66.test.eq('cameraPR (primary)')].iloc[0]
    POSTHOC_ROWS.append({'dataset': d['title'], 'test': '66 · cameraPR on rank-normal raw T_spatial', 'implied a': row66['implied a'],
                         'implied ρ': row66['implied ρ'], 'implied median VIF': np.nan, 'gene-level λ under relabeling': '—'})
    for r in (r1, r2):
        table = TESTS[ds]['cameraPR'][r].set_index('pathway_id')
        for pathway in sorted(N_CALLS[ds]['cameraPR'][r], key=lambda p: table.fdr[p]):
            members = STAT[ds]['primary'][r].z.reindex(d['sets'][pathway])
            RELABELED_ROWS.append({'dataset': d['title'], 'relabeling': SHORT[ds][r], 'pathway': NAMES[ds][pathway],
                                   'genes': len(members), 'mean z': members.mean(), 'cameraPR FDR': table.fdr[pathway],
                                   'called in the condition split': pathway in N_CALLS[ds]['cameraPR'][condition]})
POSTHOC = pd.DataFrame(POSTHOC_ROWS)
RELABELED = pd.DataFrame(RELABELED_ROWS)
POSTHOC.to_csv(tables / 'posthoc_relabeled_inflation.csv', index=False)
RELABELED.to_csv(tables / 'posthoc_relabeled_calls.csv', index=False)
display(POSTHOC.set_index(['dataset', 'test']).round(4))
if len(RELABELED):
    print('Relabeled calls of the primary, and whether the condition split calls them too:')
    print(RELABELED.groupby(['dataset', 'relabeling'])['called in the condition split'].agg(['size', 'sum']).rename(
        columns={'size': 'relabeled calls', 'sum': 'also condition calls'}))
    with pd.option_context('display.max_colwidth', 60):
        display(RELABELED.set_index(['dataset', 'relabeling', 'pathway']).round(3))

## 7 · Overlap with notebook 66's methods

The condition-split calls of the new primary are compared with each call set of notebook 66, read
from its `pathway_detail_*.csv` tables:
- cameraPR and GSEA on rank-normal raw `T_spatial`;
- our NB joint test;
- Caleb's ORA (his default null);
- his level-based set-curve calls.

The new GSEA and ORA are compared with the new primary as well. Each pair is counted over pathways and
over the redundancy clusters they touch, with the Jaccard index.

In [ ]:
DETAIL66 = {ds: pd.read_csv(nb66 / 'tables' / f'pathway_detail_{ds}.csv', index_col='pathway_id').reindex(list(DATA[ds]['sets']))
            for ds in DATASETS}
CALLS66 = {'66 cameraPR': 'called: A cameraPR', '66 GSEA': 'called: A GSEA', '66 joint test': 'called: reference joint',
           '66 B ORA': 'called: B ORA', '66 B set curve': 'called: B set curve'}


def call_set(ds, key):
    if key in CALLS66:
        return set(DETAIL66[ds].index[DETAIL66[ds][CALLS66[key]].astype(bool)])
    return N_CALLS[ds][key][list(PARTITIONS[ds])[0]]


for ds in DATASETS:   # guard: notebook 66's detail tables agree with its method table
    rows = TABLE66[TABLE66.dataset.eq(DATA[ds]['title'])].set_index('method').calls
    assert len(call_set(ds, '66 cameraPR')) == int(rows['A · cameraPR (primary)'])
    assert len(call_set(ds, '66 joint test')) == int(rows['Reference · NB joint test, relabeling-fitted VIF (ours)'])
    assert len(call_set(ds, '66 B ORA')) == int(rows['B · ORA on gene groups and clusters (his default null)'])

LABEL = {'cameraPR': 'N cameraPR', 'GSEA': 'N GSEA', 'ORA': 'N ORA', **{k: k for k in CALLS66}}
PAIRS = [('cameraPR', k) for k in CALLS66] + [('cameraPR', 'GSEA'), ('cameraPR', 'ORA')]
OVERLAP_ROWS = []
for ds in DATASETS:
    for left, right in PAIRS:
        for unit, collapse in (('pathways', lambda s, ds=ds: s), ('redundancy clusters', lambda s, ds=ds: representatives(ds, s))):
            x, y = collapse(call_set(ds, left)), collapse(call_set(ds, right))
            OVERLAP_ROWS.append({'dataset': DATA[ds]['title'], 'comparison': f'{LABEL[left]} vs {LABEL[right]}', 'unit': unit,
                                 'left': len(x), 'right': len(y), 'left only': len(x - y), 'both': len(x & y),
                                 'right only': len(y - x), 'Jaccard': len(x & y) / len(x | y) if x | y else np.nan})
OVERLAP = pd.DataFrame(OVERLAP_ROWS)
OVERLAP.to_csv(tables / 'overlap_with_notebook66.csv', index=False)
display(OVERLAP.pivot_table(index=['dataset', 'comparison'], columns='unit',
                            values=['left only', 'both', 'right only', 'Jaccard'], sort=False).round(3))

PAIR_COLOR = {'cameraPR': N_COLOR, 'GSEA': N_COLOR, 'ORA': N_COLOR, '66 cameraPR': A_COLOR, '66 GSEA': A_COLOR,
              '66 joint test': REF_COLOR, '66 B ORA': B_COLOR, '66 B set curve': B_COLOR}
fig, axes = plt.subplots(1, 2, figsize=(180 * MM, 52 * MM), sharey=True, gridspec_kw={'wspace': .08})
for ax, ds in zip(axes, DATASETS):
    rows = OVERLAP[OVERLAP.dataset.eq(DATA[ds]['title']) & OVERLAP.unit.eq('redundancy clusters')].reset_index(drop=True)
    widest = max(1, rows[['left only', 'both', 'right only']].sum(axis=1).max())
    for i, r in rows.iterrows():
        y = len(rows) - 1 - i
        left, right = PAIRS[i]
        start = 0
        for value, color in zip((r['left only'], r['both'], r['right only']), (PAIR_COLOR[left], INK, PAIR_COLOR[right])):
            ax.barh(y, value, left=start, color=color, height=.6, edgecolor='white', linewidth=.8)
            if value >= .045 * widest:
                ax.text(start + value / 2, y, str(int(value)), ha='center', va='center', fontsize=5,
                        color='white' if color in (INK, A_COLOR, REF_COLOR, N_COLOR) else INK)
            start += value
        ax.text(start + .01 * widest, y, 'J = —' if pd.isna(r['Jaccard']) else f"J = {r['Jaccard']:.2f}", va='center',
                fontsize=5.2, color=MUTED)
    ax.set_yticks(range(len(rows)), [f'{LABEL[a]} | both | {LABEL[b]}' for a, b in PAIRS][::-1])
    ax.set_xlabel('redundancy clusters called (condition split)')
    ax.set_title(DATA[ds]['title'], loc='left')
    ax.margins(x=.14)
save(fig, 'fig4_overlap')
plt.show()

## 8 · What the normalization changed at gene level

**Definitions, fixed before any result of this section:**
- Genes are ranked by raw `T_spatial` and by the primary z, both on the condition split. "Top 10%"
  means a rank percentile above 0.9.
- **Drop out:** in the raw top 10% but not in the new top 10%. **Enter:** the reverse.
- For each group of genes the table gives:
  - the median mean expression (equal-specimen log1p CP10k) and the median log mean count;
  - the share nearly absent (detected in < 1% of one group's structures);
  - the share with zero counts in a specimen;
  - the median s² = LR_spec / df₂ and the median s²_post / s₀²;
  - the median **specimen agreement**.
- **Specimen agreement** asks whether two independent pairs of specimens show the same difference
  shape. There are two disjoint pairings: (case 1 − reference 1, case 2 − reference 2) and
  (case 1 − reference 2, case 2 − reference 1). Specimens are sorted by name within each group. For
  each pairing, take the Pearson correlation over the common grid between the two pair differences of
  the specimen curves. The gene's agreement is the mean of the two correlations. Pearson removes each
  difference's mean, so offsets do not count; 1 means identical shapes, 0 unrelated, −1 opposite.
- The Spearman correlation between the new z and raw `T_spatial` is given for every partition and
  variant.

In [ ]:
def pair_agreement(curves, case, reference):
    """Mean over the two disjoint case-reference pairings of the grid Pearson correlation between the pairing's two
    specimen differences (genes,)."""
    (c1, c2), (r1, r2) = sorted(case), sorted(reference)
    values = []
    for (a, b), (x, y) in (((c1, r1), (c2, r2)), ((c1, r2), (c2, r1))):
        u, v = curves[a] - curves[b], curves[x] - curves[y]
        u, v = u - u.mean(axis=1, keepdims=True), v - v.mean(axis=1, keepdims=True)
        with np.errstate(invalid='ignore', divide='ignore'):
            values.append((u * v).sum(axis=1) / np.sqrt((u * u).sum(axis=1) * (v * v).sum(axis=1)))
    return np.mean(values, axis=0)


SPEARMAN_ROWS = []
for ds in DATASETS:
    for label, model in MODELS[ds].items():
        for variant in VARIANTS:
            frame = STAT[ds][variant][label]
            SPEARMAN_ROWS.append({'dataset': DATA[ds]['title'], 'partition': SHORT[ds][label], 'variant': variant,
                                  'Spearman(new z, raw T_spatial)': float(spearmanr(frame.z, model.statistics.T_spatial.reindex(frame.index))[0])})
SPEARMAN = pd.DataFrame(SPEARMAN_ROWS)
SPEARMAN.to_csv(tables / 'spearman_new_vs_raw.csv', index=False)
display(SPEARMAN.pivot_table(index=['dataset', 'partition'], columns='variant', values='Spearman(new z, raw T_spatial)',
                             sort=False).round(3))

GENES, CHANGE_ROWS = {}, []
for ds, d in DATA.items():
    model, new = MODELS[ds]['species'], STAT[ds]['primary']['species']
    g = pd.DataFrame(index=pd.Index(d['genes'], name='gene'))
    g['T_spatial'] = model.statistics.T_spatial
    g['F'], g['z'], g['p'], g['q'] = new.F, new.z, new.p, new.q
    g['s2'], g['s2_prior'], g['s2_post'] = new.s2, new.s2_prior, new.s2_post
    g['raw percentile'], g['new percentile'] = g.T_spatial.rank(pct=True), g.z.rank(pct=True)
    g['raw top'], g['new top'] = g['raw percentile'] > 1 - TOP_SHARE, g['new percentile'] > 1 - TOP_SHARE
    g['change'] = np.select([g['raw top'] & ~g['new top'], ~g['raw top'] & g['new top'], g['raw top'] & g['new top']],
                            ['drop out', 'enter', 'both top 10%'], 'neither')
    g['mean expression'], g['log mean count'] = d['mean_expression'], d['log_mean_count']
    g['nearly absent'], g['zero in a specimen'] = d['nearly_absent'], d['zero_in_a_specimen']
    g['log ratio'] = d['log_ratio']
    g['specimen agreement'] = pair_agreement(model.specimen_curves, d['case_specimens'], d['reference_specimens'])
    g['converged'], g['separated'] = new.converged, new.separated
    GENES[ds] = g
    for name, mask in (('drop out (raw top 10% only)', g.change.eq('drop out')), ('enter (new top 10% only)', g.change.eq('enter')),
                       ('both top 10%', g.change.eq('both top 10%')), ('all genes', pd.Series(True, index=g.index))):
        rows = g[mask]
        CHANGE_ROWS.append({'dataset': d['title'], 'genes': name, 'n': len(rows),
                            'median mean expression (log1p CP10k)': rows['mean expression'].median(),
                            'median log mean count': rows['log mean count'].median(),
                            'share nearly absent': rows['nearly absent'].mean(), 'share zero in a specimen': rows['zero in a specimen'].mean(),
                            'median |log ratio|': rows['log ratio'].abs().median(), 'median s²': rows.s2.median(),
                            'median s²_post / s₀²': (rows.s2_post / rows.s2_prior).median(),
                            'median specimen agreement': rows['specimen agreement'].median()})
    movers = g[g.change.isin(['drop out', 'enter'])].assign(shift=lambda t: t['new percentile'] - t['raw percentile'])
    movers.sort_values(['change', 'shift'], ascending=[True, True]).to_csv(tables / f'top10_movers_{ds}.csv')
    g.to_csv(tables / f'gene_change_{ds}.csv')
CHANGE = pd.DataFrame(CHANGE_ROWS)
CHANGE.to_csv(tables / 'top10_change_summary.csv', index=False)
display(CHANGE.set_index(['dataset', 'genes']).round(3))

fig, axes = plt.subplots(1, 2, figsize=(150 * MM, 64 * MM), gridspec_kw={'wspace': .3})
for ax, ds in zip(axes, DATASETS):
    g = GENES[ds]
    points = ax.scatter(g['raw percentile'], g['new percentile'], c=g['mean expression'], cmap='viridis', s=1.2,
                        linewidth=0, rasterized=True, vmin=0, vmax=np.quantile(g['mean expression'], .99))
    for v in (1 - TOP_SHARE,):
        ax.axvline(v, color=INK, lw=.5, ls='--')
        ax.axhline(v, color=INK, lw=.5, ls='--')
    counts = g.change.value_counts()
    ax.text(.995, .02, f"drop out\n{counts.get('drop out', 0):,}", transform=ax.transAxes, ha='right', va='bottom', fontsize=5.2)
    ax.text(.5, .995, f"enter {counts.get('enter', 0):,}", transform=ax.transAxes, ha='center', va='top', fontsize=5.2)
    rho = float(spearmanr(g.z, g.T_spatial)[0])
    ax.set_title(f"{DATA[ds]['title']} · condition split\nSpearman(new z, raw T_spatial) = {rho:.2f}", loc='left')
    ax.set_xlabel('rank percentile, raw T_spatial')
    ax.set_ylabel('rank percentile, normalized z (primary)')
bar = fig.colorbar(points, ax=axes, shrink=.8, pad=.02)
bar.set_label('mean expression (log1p CP10k)')
save(fig, 'fig5_raw_vs_normalized_ranks')
plt.show()

### 8.1 · Worked examples: the rule, fixed before plotting

For each dataset, on the condition split, among genes whose fits converged and are not separated:
1. **Largest drop:** among the genes that drop out of the top 10%, the one whose rank percentile
   falls most from raw `T_spatial` to the new z. Ties go to the higher raw `T_spatial`, then to the
   name.
2. **Largest gain:** among the genes that enter the top 10%, the one whose rank percentile rises
   most. Ties go to the higher new z, then to the name.

**Each panel:**
- **Top:** the four specimen curves (NB log rate per 10k counts, natural log) on the common grid.
  Case specimens are red and reference specimens blue; the second specimen of each group is dashed.
- **Bottom:** the two disjoint pairings' specimen differences, each centred on its own mean (the
  shape that `T_spatial` and LR_spec see).
- **Text:** the ranks, `T_spatial`, s² against its prior, and the specimen agreement.

In [ ]:
EXAMPLES = []
for ds in DATASETS:
    g = GENES[ds].reset_index()
    ok = g.converged & ~g.separated
    drop = g[ok & g.change.eq('drop out')].assign(move=lambda t: t['raw percentile'] - t['new percentile'])
    gain = g[ok & g.change.eq('enter')].assign(move=lambda t: t['new percentile'] - t['raw percentile'])
    EXAMPLES.append((ds, 'largest drop', drop.sort_values(['move', 'T_spatial', 'gene'], ascending=[False, False, True]).gene.iloc[0]))
    EXAMPLES.append((ds, 'largest gain', gain.sort_values(['move', 'z', 'gene'], ascending=[False, False, True]).gene.iloc[0]))

fig, axes = plt.subplots(2, len(EXAMPLES), figsize=(180 * MM, 104 * MM), gridspec_kw={'hspace': .12, 'wspace': .42,
                                                                                      'height_ratios': [1.25, 1]})
for k, (ds, kind, gene) in enumerate(EXAMPLES):
    d, model, row = DATA[ds], MODELS[ds]['species'], GENES[ds].loc[gene]
    i = d['genes'].index(gene)
    grid, curves = model.grid, model.specimen_curves
    top, bottom = axes[0, k], axes[1, k]
    for names, color in ((d['case_specimens'], CASE_COLOR), (d['reference_specimens'], REFERENCE_COLOR)):
        for name, ls in zip(names, ('-', '--')):
            top.plot(grid, curves[name][i] + np.log(1e4), color=color, ls=ls, lw=1, label=name)
    (c1, c2), (r1, r2) = d['case_specimens'], d['reference_specimens']
    for (a, b), color, ls in (((c1, r1), N_COLOR, '-'), ((c2, r2), N_COLOR, '--'), ((c1, r2), MUTED, '-'), ((c2, r1), MUTED, '--')):
        diff = curves[a][i] - curves[b][i]
        bottom.plot(grid, diff - diff.mean(), color=color, ls=ls, lw=.9, label=f'{a} − {b}')
    bottom.axhline(0, color=LIGHT, lw=.6, zorder=0)
    top.set_title(f"{d['title'].replace(' PT', '')} · {kind}\n{gene}", loc='left', fontsize=6, pad=24,
                  color=N_COLOR if kind == 'largest gain' else INK)
    top.text(0, 1.02, f"rank pct {row['raw percentile']:.3f} (raw) → {row['new percentile']:.3f} (new)\n"
             f"T_spatial {row['T_spatial']:.1f}; F {row['F']:.2f}; s² {row['s2']:.2f} (prior {row['s2_prior']:.2f})\n"
             f"specimen agreement {row['specimen agreement']:.2f}", transform=top.transAxes, fontsize=4.8, va='bottom',
             color=MUTED)
    top.set_xticklabels([])
    bottom.set_xlabel('position along the PT')
    if k == 0:
        top.set_ylabel('log rate per 10k (ln)')
        bottom.set_ylabel('pair difference − its mean (ln)')
    if k % 2 == 0:
        handles = top.get_legend_handles_labels()[0] + bottom.get_legend_handles_labels()[0]
        bottom.legend(handles=handles, loc='upper left', bbox_to_anchor=(0, -.32), ncol=2, fontsize=4.8, handlelength=1.8,
                      columnspacing=1.2)
save(fig, 'fig6_worked_gene_examples')
plt.show()
display(pd.DataFrame([{'dataset': DATA[ds]['title'], 'example': kind, 'gene': gene,
                       **GENES[ds].loc[gene, ['T_spatial', 'F', 'z', 'raw percentile', 'new percentile', 's2', 's2_prior',
                                              'mean expression', 'log ratio', 'specimen agreement']].to_dict()}
                      for ds, kind, gene in EXAMPLES]).set_index(['dataset', 'example']).round(3))

## 9 · The pathways the primary calls

For each dataset, the primary's condition-split calls, sorted by cameraPR FDR. Up to 15 are shown
here; the full list is saved. For each pathway:
- **genes:** tested members; **BH ≤ 0.05:** members significant on the primary F; **mean z:** the
  members' mean z;
- **direction:** the direction of the members' offset, from the median of each gene's mean NB log
  rate ratio (case / reference) over the grid, with the share of members higher in the case group.
  The F itself has no direction: it measures shape;
- **also called in 66 by:** notebook 66's call sets that contain the pathway.

In [ ]:
CALLED = {}
for ds, d in DATA.items():
    condition = list(PARTITIONS[ds])[0]
    test = TESTS[ds]['cameraPR'][condition].set_index('pathway_id')
    called = test[test.call.astype(bool)].sort_values(['fdr', 'p'])
    if called.empty:
        print(f"{d['title']}: the primary calls no pathway.")
        continue
    g = GENES[ds]
    rows = []
    for pathway, r in called.iterrows():
        members = g.loc[d['sets'][pathway]]
        median_ratio = members['log ratio'].median()
        rows.append({'pathway': NAMES[ds][pathway], 'library': pathway.split('::', 1)[0], 'genes': len(members),
                     'BH ≤ 0.05': int(members.q.le(GENE_FDR).sum()), 'mean z': members.z.mean(),
                     'direction': f"{'higher' if median_ratio > 0 else 'lower'} in {d['case_name']}",
                     'median log ratio': median_ratio, f"share higher in {d['case_name']}": float((members['log ratio'] > 0).mean()),
                     'cameraPR FDR': r.fdr, 'redundancy cluster': NAMES[ds][REDUNDANCY[ds].representative[pathway]],
                     'also called in 66 by': ', '.join(k for k in CALLS66 if pathway in call_set(ds, k)) or '—'})
    CALLED[ds] = pd.DataFrame(rows, index=pd.Index(called.index, name='pathway_id'))
    CALLED[ds].to_csv(tables / f'primary_calls_{ds}.csv')
    print(f"{d['title']}: {len(called)} pathways in {len(representatives(ds, set(called.index)))} redundancy clusters")
    with pd.option_context('display.max_colwidth', 60):
        shown = CALLED[ds].head(15).set_index('pathway').round(3)
        shown['cameraPR FDR'] = CALLED[ds]['cameraPR FDR'].head(15).map(lambda v: f'{v:.1e}').to_numpy()
        display(shown)

## 10 · Readings of the protocol

Where the protocol left a choice, the most literal reading was taken. Each was fixed before any fit.
- **`robust = TRUE` keeps the primary's covariate.** The protocol lists "no covariate" and
  "`robust = TRUE`" as two separate sensitivities, so only one setting changes in each.
- **Dropping the zero-count genes drops them from the whole analysis.** squeezeVar is refitted on the
  remaining genes. The gene sets are re-filtered to 10–300 remaining members, so this row tests fewer
  pathways (printed with the method table); its relabeling rule uses its own number of tested
  pathways.
- **Non-converged genes.** LR_spec is replaced by the converged genes' median when the gene's α,
  M_full or M_spec fit did not converge (nb_gam's rule). `T_spatial` is filled exactly as
  `standard_gene_model` fills it. The F is computed from the filled values. Separated genes are
  flagged and kept.
- **Both tails of z are kept.** z = Φ⁻¹(1 − p) is computed in log space at both ends.
  - Genes with almost no positional information have F near 0. These are mostly separated genes,
    absent from one group, with `T_spatial` ≈ 10⁻⁶. Their p lies within 10⁻²⁰ of 1, so they get a
    strongly negative z rather than −∞. The protocol's z is used as it stands; it is not truncated.
  - **F = 0.** A `T_spatial` truncated at 0 gives p = 1 exactly and z = −∞, which no mean-based set
    test accepts. Such a gene is tied with the smallest finite z (`moderated_f`). One gene per
    relabeling of human vs mouse is affected (section 3, "genes F = 0").
- **ORA.** gseapy applies its BH only over the pathways that hold at least one listed gene. A pathway
  without a listed gene gets p = 1 and is not called. For calibration the ORA p of the one gene list
  is used, which is notebook 61/66's list-based convention (best list p × number of lists) with one
  list.
- **Covariate.** "Log mean count per structure" is read as the log of the gene's mean raw count over
  all structures of the dataset, unweighted. It is the same in all partitions.
- **Reused, not recomputed.** The redundancy clusters and the comparison rows come from notebook
  66's saved tables. The numerator, α and specimen curves are refitted and checked against notebook
  66's cache.
- **Not a choice: what the relabelings' denominator holds.** With 2 + 2 specimens, a relabeling's
  LR_spec is the other relabeling's shape (section 2).
- **limma's warning.** "Zero sample variances detected, have been offset away from zero" refers to
  genes with LR_spec = 0. limma offsets them only to fit the prior; their own s² stays 0, and
  s²_post = d₀ s₀² / (df₂ + d₀).

## 11 · Summary

**The question.** Notebook 66's packaged set tests on `T_spatial` were not specific. Does dividing
each gene's `T_spatial` by its own specimen-to-specimen shape variation fix that? The protocol was
frozen before any fit (commit 13f0148) and was followed as written; section 10 lists the readings.

**The primary passes on human vs mouse and fails on AKI vs control.**

| Method | Human vs mouse (1,513 pathways) | AKI vs control (1,429 pathways) |
|---|---|---|
| **N · cameraPR on z (primary)** | **70 (68 clusters); 0, 0; NCDR 0; 0.044 · passes** | **93 (86 clusters); 17, 11; NCDR 0.15; 0.102 · fails calibration** |
| N · GSEA prerank on z | 77; 0, 0; 0; 0.085 · fails calibration | 51; 34, 31; 0.64; 0.140 · fails both |
| N · ORA on genes with BH ≤ 0.05 | 71; 0, 0; 0; 0.000 · passes (list-based) | 100; 8, 3; 0.06; 0.029 · passes (list-based) |
| N · cameraPR on rank-normal F | 60; 0, 0; 0; 0.044 · passes | 84; 7, 9; 0.10; 0.096 · fails calibration |
| N · squeezeVar without covariate | 76; 0, 0; 0; 0.054 · passes | 102; 32, 15; 0.23; 0.118 · fails calibration |
| N · squeezeVar `robust = TRUE` | 99; 8, 2; 0.05; 0.068 · passes | 79; 9, 10; 0.12; 0.097 · fails calibration |
| N · zero-count-specimen genes dropped | 86; 2, 0; 0.01; 0.052 · passes (1,511 pathways) | 93; 17, 11; 0.15; 0.101 · fails calibration |
| 66 · cameraPR, raw `T_spatial` | 98; 46, 29; 0.38; 0.121 · fails both | 144; 87, 86; 0.60; 0.170 · fails both |
| 66 · NB joint test (VIF fitted on the relabelings) | 32; 3, 0; 0.05; 0.067 | 12; 0, 0; 0; 0.072 |
| 66 · clustering-first ORA, single-split null | 162 and 159; 0, 0; 0.003 | 469 and 482; 2, 0; 0.005 |

Each cell gives condition calls; relabeled calls; NCDR; calibration share (limit 0.075).

**Observations.**
1. **The normalization removes most of the inflation.** On AKI, cameraPR's relabeled calls fall
   from 87 and 86 to 17 and 11, and its calibration share from 0.170 to 0.102. The primary passes the
   relabeling rule on both datasets and the calibration limit only on human vs mouse. The
   sensitivities agree: every cameraPR variant passes on human vs mouse and fails calibration on AKI
   (0.096–0.118). The protocol's decision rule needs a pass on both datasets, so it is **not met**.
   The result is reported as is.
2. **The human pass is against a weak null.** The two human "replicates" are two cortex sections of
   one donor. In human vs mouse, the denominator and the relabelings both measure section-to-section
   variation, not donor-to-donor variation. A pass there shows specificity against differences
   between sections, not between people.
3. **At gene level the moderated F is close to calibrated under relabeling; raw `T_spatial` is not.**
   - **AKI.** Share of p ≤ 0.05 is 0.050 and 0.061, with λ 0.96 and 1.00. Raw `T_spatial`'s nominal
     χ²₆ p gives 0.23 and λ ≈ 2.7.
   - **Human vs mouse.** 0.072 and 0.056, with λ 1.50 and 1.35, against 0.15 and λ ≈ 1.8.
   - **Expression.** The Spearman correlation of relabeled z with mean expression falls from
     0.17–0.22 to −0.12–0.08. d₀ is 3.5–5.6 for human vs mouse and 14–17 for AKI, so the prior's
     share of s²_post, d₀ / (df₂ + d₀), is 0.32–0.37 for human vs mouse and 0.55–0.74 for AKI. On
     AKI the denominator is mostly the expression-dependent prior.
4. **Why AKI still fails (6.1, post hoc).** Its genes are calibrated one by one, but its relabeled
   pathway z are inflated by a constant factor (a = 1.49; notebook 66's raw cameraPR: 2.13), with
   almost no growth with set size (ρ = 0.001). A larger `inter.gene.cor` would therefore not repair
   it.
   - The relabeled calls are coherent specimen programs: DNA replication, E2F targets, G1/S
     transcription, complement and coagulation, epithelial–mesenchymal transition (EMT) and ECM.
   - 6 of 17 and 6 of 11 of them are also condition calls, among them E2F targets and EMT.
   - With one specimen per condition on each side, a program that differs between the two AKI mice
     moves all its genes together. Each gene's own denominator cannot see that coherence.
   - Human vs mouse shows no such inflation (a = 0.96).
5. **The new calls are a subset of notebook 66's cameraPR calls.** Over redundancy clusters:
   - 66 of 68 (human vs mouse) and 86 of 86 (AKI) are shared with raw-`T_spatial` cameraPR
     (Jaccard 0.70 and 0.66);
   - Jaccard 0.38 and 0.14 with the NB joint test, which it nearly contains (28 of 32 and 11 of 12
     joint-test pathways);
   - Jaccard 0.14 and 0.10 with clustering-first ORA.
   
   The leading human vs mouse calls are transport and metabolism pathways: inorganic ion and amino
   acid transport, fatty acid metabolism, branched-chain amino acid degradation, xenobiotic and
   arginine/proline metabolism.
6. **What the normalization changes at gene level.** Ranks stay close to raw `T_spatial` on the
   condition split (Spearman 0.94 and 0.97). The 164 (human) and 197 (AKI) genes that drop out of
   the top 10% differ from the genes that enter:
   - they are more highly expressed (median log1p CP10k 0.73 vs 0.24, and 1.14 vs 0.27);
   - their specimens vary more in shape (median s² 2.8 vs 0.7, and 5.9 vs 1.1);
   - their specimen agreement is lower (0.87 vs 0.92, and 0.87 vs 0.93).
   
   Worked examples:
   - *Ighm* (human vs mouse) drops out: its two human sections have opposite shapes (agreement −0.26).
   - *Cyp7b1* (AKI) drops out: its large offset comes with specimen shapes that vary 23 times more
     than its prior.
   - *Nop2* and *Zc3h12c* enter: small, reproducible shapes (agreement 0.90 and 0.96).

**What this cannot tell you.** The relabelings mix the two conditions, so they cannot see an
artefact that follows the condition. With two specimens per group every result is descriptive, and
on AKI the per-gene denominator is mostly the prior (share 0.55–0.74). Section 6.1 was added
after the method tables were seen.

In [ ]:
for ds in DATASETS:
    row = METHOD_TABLE[METHOD_TABLE.dataset.eq(DATA[ds]['title']) & METHOD_TABLE.method.eq(N_METHODS['cameraPR'])].iloc[0]
    print(f"{DATA[ds]['title']}: the primary (cameraPR on the z of the specimen-normalized F) "
          f"{'PASSES' if PASS[ds] else 'FAILS'} the frozen pass rule. Calls {row['calls']} "
          f"({row['calls (redundancy clusters)']} redundancy clusters); relabeled {row['relabeled calls']}; "
          f"NCDR {row['NCDR']:.2f} ({row['relabeling rule']}); calibration share {row['calibration share']:.3f} "
          f"({row['calibration']}).")
print('The two human specimens are two cortex sections of one donor: the human vs mouse relabelings pair sections, '
      'not donors.')
print('Decision rule (protocol): a candidate replacement for the joint test only if the primary passes on both datasets: '
      + ('met' if all(PASS.values()) else 'not met') + '.')

## 12 · Run record

In [ ]:
import platform

from rpy2.robjects import r as R

cache_entries = []
for sidecar in sorted(cache.glob('*.key.json'), key=lambda p: json.loads(p.read_text())['written_utc']):
    meta = json.loads(sidecar.read_text())
    if meta['stage'] in SECONDS:
        cache_entries.append({'stage': meta['stage'], 'key': meta['key'], 'written_utc': meta['written_utc'],
                              'compute seconds': round(SECONDS[meta['stage']], 1)})
CACHE_TABLE = pd.DataFrame(cache_entries).drop_duplicates('stage', keep='last').sort_values('stage').reset_index(drop=True)
display(CACHE_TABLE)
record = {
    'notebook_logic_version': NOTEBOOK_LOGIC_VERSION, 'protocol': 'docs/results/pt-gene-model-nb.md, Notebook 67 protocol '
    '(commit 13f0148)', 'datasets': INPUT_TABLE.to_dict(), 'n_jobs': N_JOBS,
    'gene_model': {'module': 'pseudospace.pathway_pipelines.standard_gene_model', 'basis_df': BASIS_DF,
                   'grid_points': GRID_POINTS, 'nb_gam': nb_gam.GENE_MODEL},
    'statistic': {'numerator': 'T_spatial (NB LR, df1 = 6)', 'denominator': 'LR_spec = D(M_full) - D(M_spec), '
                  'pathway_pipelines.specimen_shape_lr', 'df2': DF2, 'moderation': 'limma::squeezeVar via '
                  'pathway_pipelines.squeeze_var', 'covariate': 'log mean count per structure', 'variants': VARIANTS,
                  'z': 'Phi^-1(1 - p), log space (pathway_pipelines.moderated_f)'},
    'pathway_tests': {'cameraPR': {'inter.gene.cor': CAMERA_COR, 'use.ranks': False},
                      'gsea': {'permutations': GSEA_PERMUTATIONS, 'seed': GSEA_SEED, 'metric': 'raw z'},
                      'ora': {'gene_fdr': GENE_FDR, 'background': 'all tested genes', 'function': 'gseapy.enrich'},
                      'set_sizes': SET_SIZES,
                      'ora_gene_lists': {ds: {SHORT[ds][p]: len(v) for p, v in ORA_LISTS[ds].items()} for ds in DATASETS},
                      'pathways_without_zero_count_genes': {ds: len(SETS_KEPT[ds]) for ds in DATASETS}},
    'pass_rule': {'ncdr_max': NCDR_MAX, 'null_rate': NULL_RATE, 'calibration_max': CALIBRATION_MAX},
    'primary_passes': PASS, 'methods': METHOD_TABLE.to_dict(orient='records'),
    'moderation': SQUEEZE.to_dict(orient='records'), 'denominator': DENOM_TABLE.to_dict(orient='records'),
    'gene_checks': GENE_CHECKS.to_dict(orient='records'), 'overlap': OVERLAP.to_dict(orient='records'),
    'top10_change': CHANGE.to_dict(orient='records'), 'spearman_new_vs_raw': SPEARMAN.to_dict(orient='records'),
    'worked_examples': [{'dataset': ds, 'example': kind, 'gene': gene} for ds, kind, gene in EXAMPLES],
    'guards': guard_log,
    'versions': {'python': platform.python_version(), 'R': str(R('R.version.string')[0]),
                 'limma': str(R('as.character(packageVersion("limma"))')[0]),
                 **{m: version(m) for m in ('numpy', 'scipy', 'pandas', 'gseapy', 'rpy2', 'joblib', 'matplotlib')}},
    'stage_cache': CACHE_TABLE.to_dict(orient='records'), 'compute_seconds_total': float(sum(SECONDS.values())),
    'wall_seconds': time.perf_counter() - NOTEBOOK_START}
(output / 'run_record.json').write_text(json.dumps(record, indent=2, default=str))
print(f"Wall time {record['wall_seconds']:.0f} s; summed uncached compute time of the cached stages "
      f"{record['compute_seconds_total']:.0f} s")